# Hate speech detection with fine-tuned BERT: revised experiments (Kaggle version)

Runs every experiment the reviewers asked for: fixed stratified 70/10/20 split, 5 seeds, training-set class weights, early stopping on validation macro-F1, 7 models, 3 ablations, HateXplain and HateBR generalisation, cost table, significance tests, error-analysis sheet. Its outputs (`results/`, `generated/`) are what the revised manuscript includes.

## Before you start (Kaggle settings, right-hand panel)
1. **Accelerator: GPU T4 x2** (or P100). Either works; the code uses one GPU.
2. **Internet: On** (needed to download the model weights and the datasets; Kaggle asks you to verify your phone number once).
3. Optional: *Add Input -> Upload* your `labeled_data.csv` as a dataset. If it is not there the notebook downloads the identical file from the authors' GitHub and checks its SHA-256.

## Two ways to run
* **Recommended: Save Version -> Save & Run All (Commit).** It runs in the background (you can close the tab), takes about 5 hours on a T4, and everything in `/kaggle/working` is kept in the notebook's **Output** tab. Kaggle's limit is 12 h per run and 30 GPU-hours per week.
* **Interactive:** run the cells top to bottom. The session is deleted when you leave, so download `hs_results_and_generated.zip` (last cell) before you close it.

## If a run stops half-way
Download the zip, upload it as a Kaggle dataset, add it as an input of the notebook and run again: finished runs are found and skipped automatically. `QUICK = True` gives 3 seeds instead of 5 (about 2.5 h); the paper text says five seeds, so use `False` for the final run.

The error-analysis coding (two people) is done outside this notebook: download `generated/errors_to_code.csv` from the Output tab.

## 0. Setup

In [ ]:
import os, sys, glob, shutil, json
import torch, transformers, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> set Accelerator to GPU in the right-hand panel!")
# internet check (model weights come from huggingface.co)
import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10); print("internet: OK")
except Exception as e:
    print("internet: NOT available -> switch Internet to On in the right-hand panel", e)

In [ ]:
# ---- settings you may change ----
QUICK = False                       # True: 3 seeds instead of 5
SEEDS = [13, 21, 42] if QUICK else [13, 21, 42, 87, 101]
WORK  = "/kaggle/working/hs_revision"
for d in ("data_cache", "results", "generated"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)

# 1) your own copy of the Davidson file, if you uploaded it as a dataset
hits = glob.glob("/kaggle/input/**/labeled_data.csv", recursive=True)
dst = WORK + "/data_cache/davidson_labeled_data.csv"
if hits and not os.path.exists(dst):
    shutil.copy(hits[0], dst); print("using your dataset file:", hits[0])
elif not hits:
    print("no labeled_data.csv among the inputs -> it will be downloaded from the authors' GitHub")

# 2) results of an earlier, interrupted run (a zip you uploaded as a dataset) are restored so they are skipped
for r in glob.glob("/kaggle/input/**/results", recursive=True):
    if os.path.isdir(r):
        shutil.copytree(r, WORK + "/results", dirs_exist_ok=True); print("restored earlier results from", r)
# runs made with an older version of the code (different test set) are removed so they are re-done
for p in glob.glob(WORK + "/results/davidson/*/seed*/metrics.json"):
    if json.load(open(p))["n_test"] != 4953:
        shutil.rmtree(os.path.dirname(p)); print("removed outdated run", p)
n_done = len(glob.glob(WORK + "/results/*/*/seed*/metrics.json"))
print("finished runs found:", n_done, "| SEEDS =", SEEDS)
os.makedirs("/kaggle/working/hs", exist_ok=True)
%cd /kaggle/working/hs

## 1. Code of the experiment package
The next four cells write the same files that are in the repository (`hs_common.py`, `run_experiments.py`, `aggregate_results.py`, `check_claims.py`), i.e. the implementation of Section 3 of the paper. Do not edit them unless you also change the paper.

In [ ]:
%%writefile hs_common.py
"""Shared data loading, preprocessing, splitting and metric code.

Everything that turns raw files into numbers lives here, so that all models are
trained, evaluated and reported through exactly the same code path.
"""
import hashlib
import html
import json
import os
import re
import sys
import urllib.request

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

LABELS = ["Hate Speech", "Offensive Language", "Clean Text"]  # ids 0, 1, 2
SPLIT_SEED = 42  # one fixed split for every model and every training seed

DAVIDSON_URL = ("https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/"
                "master/data/labeled_data.csv")
DAVIDSON_SHA256 = "fcb8bc7c68120ae4af04a5b9acd58585513ede11e1548ebf36a5c2040b6f6281"
HATEXPLAIN_URL = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/dataset.json"
HATEXPLAIN_SPLIT_URL = "https://raw.githubusercontent.com/hate-alert/HateXplain/master/Data/post_id_divisions.json"
HATEBR_URL = "https://raw.githubusercontent.com/franciellevargas/HateBR/main/dataset/HateBR.csv"

# Small normalisation dictionary for common social-media abbreviations (the full list is this one).
SLANG = {"u": "you", "ur": "your", "r": "are", "gr8": "great", "b4": "before", "thx": "thanks",
         "pls": "please", "plz": "please", "idk": "i do not know", "imo": "in my opinion",
         "omg": "oh my god", "cuz": "because", "bc": "because", "tho": "though", "ya": "you",
         "smh": "shaking my head", "tbh": "to be honest"}


def download(url, path):
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        print(f"downloading {url}", file=sys.stderr)
        urllib.request.urlretrieve(url, path)
    return path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def clean_text(t, keep_social=False):
    """Preprocessing used for all English data.

    Default: remove URLs, @mentions, #hashtags, emoji and special characters, lowercase, and
    expand the abbreviations in SLANG.  keep_social=True is the ablation that keeps hashtag
    words, replaces mentions by '@user' and URLs by 'http', and keeps emoji.
    """
    t = html.unescape(str(t))
    t = re.sub(r"^\s*rt\s+", "", t, flags=re.I)
    if keep_social:
        t = re.sub(r"https?://\S+|www\.\S+", "http", t)
        t = re.sub(r"@\w+", "@user", t)
        t = re.sub(r"#(\w+)", r"\1", t)
        t = t.lower()
        t = "".join(c if (c.isalnum() or c.isspace() or c in "@'!?.," or ord(c) >= 0x2190) else " " for c in t)
    else:
        t = re.sub(r"https?://\S+|www\.\S+", "", t)
        t = re.sub(r"@\w+", "", t)
        t = re.sub(r"#\w+", "", t)
        t = t.lower()
        t = re.sub(r"[^a-z0-9\s']", " ", t)
    return " ".join(SLANG.get(w, w) for w in t.split())


def clean_pt(t):
    t = html.unescape(str(t))
    t = re.sub(r"https?://\S+|www\.\S+", "", t)
    t = re.sub(r"@\w+", "@user", t)
    return " ".join(t.lower().split())


def load_davidson(cache_dir, keep_social=False):
    path = download(DAVIDSON_URL, os.path.join(cache_dir, "davidson_labeled_data.csv"))
    digest = sha256(path)
    if digest != DAVIDSON_SHA256:
        print(f"WARNING: Davidson file hash {digest} differs from the one used in the paper", file=sys.stderr)
    raw = pd.read_csv(path)
    df = pd.DataFrame({
        "id": raw.iloc[:, 0].values,
        "raw": raw["tweet"].values,
        "y": raw["class"].values.astype(int),  # 0 hate, 1 offensive, 2 neither (= clean)
        "n_votes": raw["count"].values,
        "v_hate": raw["hate_speech"].values,
        "v_off": raw["offensive_language"].values,
        "v_clean": raw["neither"].values,
    })
    n_all = len(df)
    # The row set (and hence the split) is defined by the default cleaning, so that every variant, including the
    # keep-social ablation, is trained and tested on exactly the same posts.
    keep_rows = np.array([len(clean_text(t, False)) > 0 for t in df["raw"]])
    df = df[keep_rows].reset_index(drop=True)
    df["text"] = [clean_text(t, keep_social) for t in df["raw"]]
    assert (df["text"].str.len() > 0).all(), "a post became empty under the keep-social cleaning"
    info = {"file_sha256": digest, "rows_in_file": int(n_all), "rows_empty_after_cleaning": int(n_all - len(df)),
            "rows_used": int(len(df))}
    return df, info


def load_hatexplain(cache_dir):
    data = json.load(open(download(HATEXPLAIN_URL, os.path.join(cache_dir, "hatexplain_dataset.json")), encoding="utf8"))
    div = json.load(open(download(HATEXPLAIN_SPLIT_URL, os.path.join(cache_dir, "hatexplain_divisions.json")), encoding="utf8"))
    lab = {"hatespeech": 0, "offensive": 1, "normal": 2}
    split_of = {pid: s for s, ids in div.items() for pid in ids}
    rows, dropped = [], 0
    for pid, rec in data.items():
        votes = [lab[a["label"]] for a in rec["annotators"]]
        counts = np.bincount(votes, minlength=3)
        if counts.max() < 2:  # three different labels: no majority, dropped as in the original paper
            dropped += 1
            continue
        rows.append({"id": pid, "y": int(counts.argmax()), "text": clean_text(" ".join(rec["post_tokens"])),
                     "split": {"val": "val"}.get(split_of.get(pid, "train"), split_of.get(pid, "train"))})
    df = pd.DataFrame(rows)
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df, {"posts_without_majority_dropped": dropped, "rows_used": int(len(df))}


def load_hatebr(cache_dir):
    raw = pd.read_csv(download(HATEBR_URL, os.path.join(cache_dir, "hatebr.csv")))
    df = pd.DataFrame({"id": raw["id"].values, "y": raw["label_final"].values.astype(int),
                       "text": [clean_pt(t) for t in raw["comentario"]]})
    df = df[df["text"].str.len() > 0].reset_index(drop=True)
    return df, {"rows_used": int(len(df))}  # binary: 1 = offensive


def split_indices(y, seed=SPLIT_SEED):
    """Stratified 70 / 10 / 20 split into train / validation / test."""
    idx = np.arange(len(y))
    tr_val, te = train_test_split(idx, test_size=0.20, stratify=y, random_state=seed)
    tr, va = train_test_split(tr_val, test_size=0.125, stratify=np.asarray(y)[tr_val], random_state=seed)
    return np.sort(tr), np.sort(va), np.sort(te)


def class_weights(y, k):
    counts = np.bincount(y, minlength=k).astype(float)
    return counts.sum() / (k * np.maximum(counts, 1))  # n / (k * n_c)


def cm_metrics(cm):
    """All reported numbers come from a confusion matrix (rows = true, columns = predicted)."""
    cm = np.asarray(cm, dtype=float)
    n = cm.sum()
    tp = np.diag(cm)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    tn = n - tp - fp - fn
    safe = lambda a, b: np.where(b > 0, a / np.maximum(b, 1e-12), 0.0)
    P, R = safe(tp, tp + fp), safe(tp, tp + fn)
    F = safe(2 * P * R, P + R)
    S = safe(tn, tn + fp)
    sup = cm.sum(1)
    w = sup / sup.sum()
    return {
        "acc": float(tp.sum() / n), "support": sup.tolist(),
        "P": P.tolist(), "R": R.tolist(), "F1": F.tolist(), "spec": S.tolist(),
        "macro": {"P": float(P.mean()), "R": float(R.mean()), "F1": float(F.mean()), "spec": float(S.mean())},
        "weighted": {"P": float((P * w).sum()), "R": float((R * w).sum()), "F1": float((F * w).sum()),
                     "spec": float((S * w).sum())},
    }


def confusion(y_true, y_pred, k):
    cm = np.zeros((k, k), dtype=np.int64)
    np.add.at(cm, (np.asarray(y_true), np.asarray(y_pred)), 1)
    return cm

In [ ]:
%%writefile run_experiments.py
#!/usr/bin/env python
"""Train and evaluate every model reported in the paper.

Example (full run, needs a GPU and internet access to the Hugging Face hub):
    python run_experiments.py --out results --seeds 13 21 42 87 101 \
        --models tfidf_lr lstm bert distilbert roberta hatebert toxicbert bert_unweighted bert_focal bert_keepsocial

Cross-dataset / cross-lingual runs (train on another dataset, or XLM-R for Portuguese):
    python run_experiments.py --out results --train-on hatexplain --models bert hatebert --seeds 13 21 42
    python run_experiments.py --out results --models xlmr --seeds 13 21 42
    python run_experiments.py --out results --train-on hatebr --models xlmr --seeds 13 21 42

Finished runs are skipped, so an interrupted run can simply be restarted.
"""
import argparse
import copy
import json
import os
import platform
import random
import sys
import time

import numpy as np
import pandas as pd

import hs_common as C

# ------------------------------------------------------------------ configuration (reported in the paper)
HP = dict(max_len=128, batch_size=32, lr=2e-5, weight_decay=0.01, warmup_ratio=0.1, max_epochs=5,
          patience=2, grad_clip=1.0, focal_gamma=2.0,
          lstm=dict(emb=128, hidden=128, dropout=0.3, max_len=64, epochs=8, batch_size=64, lr=1e-3, min_freq=2),
          tfidf=dict(ngram=(1, 2), min_df=2, C=1.0))
DEFAULT_SEEDS = [13, 21, 42, 87, 101]

MODELS = {
    "tfidf_lr": dict(kind="tfidf"),
    "lstm": dict(kind="lstm"),
    "bert": dict(kind="hf", hf="bert-base-uncased", loss="weighted", arch="bert"),
    "distilbert": dict(kind="hf", hf="distilbert-base-uncased", loss="weighted", arch="distilbert"),
    "roberta": dict(kind="hf", hf="roberta-base", loss="weighted", arch="roberta"),
    "hatebert": dict(kind="hf", hf="GroNLP/hateBERT", loss="weighted", arch="bert"),
    "toxicbert": dict(kind="hf", hf="unitary/toxic-bert", loss="weighted", arch="bert"),
    "bert_unweighted": dict(kind="hf", hf="bert-base-uncased", loss="ce", arch="bert"),
    "bert_focal": dict(kind="hf", hf="bert-base-uncased", loss="focal", arch="bert"),
    "bert_keepsocial": dict(kind="hf", hf="bert-base-uncased", loss="weighted", arch="bert", keep_social=True),
    "xlmr": dict(kind="hf", hf="xlm-roberta-base", loss="weighted", arch="roberta", multilingual=True),
}


def set_seed(seed):
    import torch
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def env_info():
    info = {"python": platform.python_version(), "platform": platform.platform(), "cpu_count": os.cpu_count()}
    try:
        import torch, transformers, sklearn
        info.update(torch=torch.__version__, transformers=transformers.__version__, sklearn=sklearn.__version__)
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only)"
    except Exception as e:  # pragma: no cover
        info["error"] = repr(e)
    return info


# ------------------------------------------------------------------ transformer models
def make_tokenizer_and_model(spec, num_labels, smoke_texts=None):
    import torch  # noqa
    from transformers import (AutoModelForSequenceClassification, AutoTokenizer, BertConfig, DistilBertConfig,
                              RobertaConfig, BertForSequenceClassification, DistilBertForSequenceClassification,
                              RobertaForSequenceClassification, PreTrainedTokenizerFast)
    if smoke_texts is None:
        tok = AutoTokenizer.from_pretrained(spec["hf"])
        model = AutoModelForSequenceClassification.from_pretrained(spec["hf"], num_labels=num_labels,
                                                                   ignore_mismatched_sizes=True)
        return tok, model
    # smoke test: tiny random model + locally trained word-piece tokenizer, no downloads
    from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
    tk = Tokenizer(models.WordPiece(unk_token="[UNK]"))
    tk.normalizer = normalizers.BertNormalizer(lowercase=True)
    tk.pre_tokenizer = pre_tokenizers.BertPreTokenizer()
    tk.train_from_iterator(smoke_texts, trainers.WordPieceTrainer(
        vocab_size=1500, special_tokens=["[PAD]", "[UNK]", "[CLS]", "[SEP]", "[MASK]"]))
    tok = PreTrainedTokenizerFast(tokenizer_object=tk, unk_token="[UNK]", pad_token="[PAD]", cls_token="[CLS]",
                                  sep_token="[SEP]", mask_token="[MASK]")
    v = tok.vocab_size
    if spec["arch"] == "distilbert":
        cfg = DistilBertConfig(vocab_size=v, dim=32, n_layers=2, n_heads=2, hidden_dim=64, num_labels=num_labels,
                               max_position_embeddings=160)
        return tok, DistilBertForSequenceClassification(cfg)
    if spec["arch"] == "roberta":
        cfg = RobertaConfig(vocab_size=v, hidden_size=32, num_hidden_layers=2, num_attention_heads=2,
                            intermediate_size=64, num_labels=num_labels, max_position_embeddings=170,
                            pad_token_id=0, bos_token_id=2, eos_token_id=3, type_vocab_size=1)
        return tok, RobertaForSequenceClassification(cfg)
    cfg = BertConfig(vocab_size=v, hidden_size=32, num_hidden_layers=2, num_attention_heads=2, intermediate_size=64,
                     num_labels=num_labels, max_position_embeddings=160)
    return tok, BertForSequenceClassification(cfg)


def batches(texts, tok, max_len, bs, order):
    for i in range(0, len(order), bs):
        idx = order[i:i + bs]
        enc = tok([texts[j] for j in idx], truncation=True, max_length=max_len, padding=True, return_tensors="pt")
        yield idx, enc


def predict_probs(model, tok, texts, max_len, device):
    import torch
    model.eval()
    lens = np.array([len(t) for t in texts])
    order = np.argsort(lens)  # sort by length: faster, identical results
    out = np.zeros((len(texts), model.config.num_labels), dtype=np.float32)
    with torch.no_grad():
        for idx, enc in batches(texts, tok, max_len, 64, order):
            enc = {k: v.to(device) for k, v in enc.items() if k != "token_type_ids" or model.config.model_type == "bert"}
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                logits = model(**enc).logits
            out[idx] = torch.softmax(logits.float(), -1).cpu().numpy()
    return out


def loss_fn(kind, logits, y, weights, gamma):
    import torch
    import torch.nn.functional as F
    if kind == "ce":
        return F.cross_entropy(logits, y)
    if kind == "weighted":
        return F.cross_entropy(logits, y, weight=weights)
    logp = F.log_softmax(logits, -1)  # focal loss, alpha = class weights
    logpt = logp.gather(1, y[:, None])[:, 0]
    pt = logpt.exp()
    return -(weights[y] * (1 - pt) ** gamma * logpt).mean()


def train_hf(spec, tr, va, seed, k, device, smoke):
    import torch
    from transformers import get_linear_schedule_with_warmup
    set_seed(seed)
    tok, model = make_tokenizer_and_model(spec, k, smoke_texts=list(tr["text"]) if smoke else None)
    model.to(device)
    max_len = HP["max_len"]
    y_tr = tr["y"].values
    w = torch.tensor(C.class_weights(y_tr, k), dtype=torch.float32, device=device)
    opt = torch.optim.AdamW(model.parameters(), lr=HP["lr"], weight_decay=HP["weight_decay"])
    steps_per_epoch = int(np.ceil(len(tr) / HP["batch_size"]))
    total = steps_per_epoch * HP["max_epochs"]
    sched = get_linear_schedule_with_warmup(opt, int(HP["warmup_ratio"] * total), total)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    texts = list(tr["text"])
    best, best_state, bad, epochs_run, t0 = -1.0, None, 0, 0, time.time()
    rng = np.random.RandomState(seed)
    for ep in range(HP["max_epochs"]):
        model.train()
        order = rng.permutation(len(texts))
        for idx, enc in batches(texts, tok, max_len, HP["batch_size"], order):
            enc = {k_: v.to(device) for k_, v in enc.items() if k_ != "token_type_ids" or model.config.model_type == "bert"}
            yb = torch.tensor(y_tr[idx], device=device)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                logits = model(**enc).logits
            loss = loss_fn(spec["loss"], logits.float(), yb, w, HP["focal_gamma"])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), HP["grad_clip"])
            scaler.step(opt)
            scaler.update()
            sched.step()
        epochs_run += 1
        pv = predict_probs(model, tok, list(va["text"]), max_len, device)
        f1 = C.cm_metrics(C.confusion(va["y"].values, pv.argmax(1), k))["macro"]["F1"]
        print(f"    epoch {ep + 1}: val macro-F1 {f1:.4f}", file=sys.stderr)
        if f1 > best:
            best, bad = f1, 0
            best_state = {n: p.detach().cpu().clone() for n, p in model.state_dict().items()}
        else:
            bad += 1
            if bad >= HP["patience"]:
                break
    train_seconds = time.time() - t0
    model.load_state_dict(best_state)
    n_params = int(sum(p.numel() for p in model.parameters()))
    return model, tok, dict(train_seconds=train_seconds, epochs_run=epochs_run, best_val_macro_f1=best,
                            n_params=n_params)


# ------------------------------------------------------------------ LSTM and TF-IDF baselines
def train_lstm(tr, va, seed, k, device, smoke):
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from collections import Counter
    h = HP["lstm"]
    set_seed(seed)
    cnt = Counter(w for t in tr["text"] for w in t.split())
    vocab = {w: i + 2 for i, (w, c) in enumerate(cnt.most_common()) if c >= h["min_freq"]}  # 0 pad, 1 unk

    def enc(texts):
        X = np.zeros((len(texts), h["max_len"]), dtype=np.int64)
        for i, t in enumerate(texts):
            ids = [vocab.get(w, 1) for w in t.split()][:h["max_len"]]
            X[i, :len(ids)] = ids
        return torch.tensor(X)

    class Net(nn.Module):
        def __init__(s):
            super().__init__()
            s.emb = nn.Embedding(len(vocab) + 2, h["emb"], padding_idx=0)
            s.lstm = nn.LSTM(h["emb"], h["hidden"], batch_first=True, bidirectional=True)
            s.drop = nn.Dropout(h["dropout"])
            s.out = nn.Linear(2 * h["hidden"], k)

        def forward(s, x):
            m = (x != 0).unsqueeze(-1).float()
            o, _ = s.lstm(s.emb(x))
            pooled = (o * m).sum(1) / m.sum(1).clamp(min=1)
            return s.out(s.drop(pooled))

    net = Net().to(device)
    Xtr, ytr = enc(list(tr["text"])), torch.tensor(tr["y"].values)
    Xva = enc(list(va["text"]))
    w = torch.tensor(C.class_weights(tr["y"].values, k), dtype=torch.float32, device=device)
    opt = torch.optim.Adam(net.parameters(), lr=h["lr"])
    best, best_state, t0 = -1, None, time.time()
    rng = np.random.RandomState(seed)
    epochs = 2 if smoke else h["epochs"]
    for ep in range(epochs):
        net.train()
        order = rng.permutation(len(Xtr))
        for i in range(0, len(order), h["batch_size"]):
            b = order[i:i + h["batch_size"]]
            loss = F.cross_entropy(net(Xtr[b].to(device)), ytr[b].to(device), weight=w)
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            pv = net(Xva.to(device)).argmax(1).cpu().numpy()
        f1 = C.cm_metrics(C.confusion(va["y"].values, pv, k))["macro"]["F1"]
        if f1 > best:
            best, best_state = f1, copy.deepcopy(net.state_dict())
    net.load_state_dict(best_state)
    info = dict(train_seconds=time.time() - t0, epochs_run=epochs, best_val_macro_f1=best,
                n_params=int(sum(p.numel() for p in net.parameters())))

    def predict(texts):
        net.eval()
        with torch.no_grad():
            return F.softmax(net(enc(texts).to(device)), -1).cpu().numpy()
    return predict, info


def train_tfidf(tr, va, seed, k):
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    h = HP["tfidf"]
    t0 = time.time()
    vec = TfidfVectorizer(ngram_range=h["ngram"], min_df=h["min_df"], sublinear_tf=True)
    X = vec.fit_transform(tr["text"])
    clf = LogisticRegression(C=h["C"], class_weight="balanced", max_iter=2000).fit(X, tr["y"])
    info = dict(train_seconds=time.time() - t0, epochs_run=1, best_val_macro_f1=None,
                n_params=int(clf.coef_.size + clf.intercept_.size))
    return (lambda texts: clf.predict_proba(vec.transform(texts))), info


# ------------------------------------------------------------------ evaluation helpers
def timed(fn, texts, device):
    t0 = time.time()
    p = fn(texts)
    return p, (time.time() - t0) / max(len(texts), 1) * 1000.0  # ms per post


def eval_cross(name, df_eval, predict, k_model):
    """Evaluate a trained model on another dataset's test set."""
    p = predict(list(df_eval["text"]))
    pred = p.argmax(1)
    if name == "hatebr":  # binary target: collapse Hate+Offensive -> offensive
        pred = (pred != 2).astype(int) if k_model == 3 else pred
        cm = C.confusion(df_eval["y"].values, pred, 2)
    else:
        cm = C.confusion(df_eval["y"].values, pred, 3)
    return {"cm": cm.tolist(), **C.cm_metrics(cm)}


def get_data(train_on, keep_social, cache, smoke):
    """Returns dict name -> (df, split-dict, info)."""
    data = {}
    df, info = C.load_davidson(cache, keep_social)
    tr, va, te = C.split_indices(df["y"].values)
    data["davidson"] = (df, {"train": tr, "val": va, "test": te}, info)
    if not keep_social:
        hx, hinfo = C.load_hatexplain(cache)
        data["hatexplain"] = (hx, {s: np.where(hx["split"].values == s)[0] for s in ("train", "val", "test")}, hinfo)
        hb, binfo = C.load_hatebr(cache)
        btr, bva, bte = C.split_indices(hb["y"].values)
        data["hatebr"] = (hb, {"train": btr, "val": bva, "test": bte}, binfo)
    if smoke:  # tiny subsets so that the whole pipeline runs in seconds on a CPU
        for name, (d, sp, inf) in list(data.items()):
            rs = np.random.RandomState(0)
            data[name] = (d, {s: np.sort(rs.choice(ix, size=min(len(ix), 600 if s == "train" else 200), replace=False))
                              for s, ix in sp.items()}, inf)
    return data


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="results")
    ap.add_argument("--cache", default="data_cache")
    ap.add_argument("--models", nargs="+", default=["tfidf_lr", "lstm", "bert", "distilbert", "roberta", "hatebert",
                                                     "toxicbert", "bert_unweighted", "bert_focal", "bert_keepsocial"])
    ap.add_argument("--seeds", nargs="+", type=int, default=DEFAULT_SEEDS)
    ap.add_argument("--train-on", default="davidson", choices=["davidson", "hatexplain", "hatebr"])
    ap.add_argument("--smoke", action="store_true", help="tiny random models, no downloads of weights (pipeline test)")
    args = ap.parse_args()

    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    env = env_info()
    os.makedirs(args.out, exist_ok=True)
    json.dump(env, open(os.path.join(args.out, "env.json"), "w"), indent=1)
    print("environment:", env, file=sys.stderr)

    cache_by_variant = {}
    for mname in args.models:
        spec = MODELS[mname]
        keep = spec.get("keep_social", False)
        if keep not in cache_by_variant:
            cache_by_variant[keep] = get_data(args.train_on, keep, args.cache, args.smoke)
        data = cache_by_variant[keep]
        if args.train_on != "davidson" and keep:
            continue
        df, sp, info = data[args.train_on]
        k = 2 if args.train_on == "hatebr" else 3
        tr, va, te = (df.iloc[sp[s]] for s in ("train", "val", "test"))
        if mname == "xlmr" or args.train_on == "hatebr":
            pass
        # English-only models cannot read Portuguese; only run them on English data
        if args.train_on == "hatebr" and not spec.get("multilingual"):
            continue
        seeds = [args.seeds[0]] if spec["kind"] == "tfidf" else args.seeds
        for seed in seeds:
            rdir = os.path.join(args.out, args.train_on, mname, f"seed{seed}")
            if os.path.exists(os.path.join(rdir, "metrics.json")):
                print(f"skip {rdir}", file=sys.stderr)
                continue
            print(f"== {args.train_on} | {mname} | seed {seed}", file=sys.stderr)
            if spec["kind"] == "hf":
                model, tok, tinfo = train_hf(spec, tr, va, seed, k, device, args.smoke)
                predict = lambda texts, m=model, t=tok: predict_probs(m, t, texts, HP["max_len"], device)
            elif spec["kind"] == "lstm":
                predict, tinfo = train_lstm(tr, va, seed, k, device, args.smoke)
            else:
                predict, tinfo = train_tfidf(tr, va, seed, k)
            ptest, ms_per_post = timed(predict, list(te["text"]), device)
            cm = C.confusion(te["y"].values, ptest.argmax(1), k)
            res = {"model": mname, "seed": seed, "train_on": args.train_on, "n_test": int(len(te)),
                   "n_train": int(len(tr)), "n_val": int(len(va)), "cm": cm.tolist(), **C.cm_metrics(cm),
                   "cost": {**tinfo, "infer_ms_per_post": ms_per_post,
                            "peak_gpu_mem_mb": (torch.cuda.max_memory_allocated() / 2 ** 20) if device == "cuda" else None},
                   "hp": {kk: vv for kk, vv in HP.items()}, "data_info": info, "env": env, "cross": {}}
            # cross-dataset evaluation (English 3-class model -> HateXplain test; multilingual -> HateBR test)
            for other in ("davidson", "hatexplain", "hatebr"):
                if other == args.train_on or other not in data:
                    continue
                odf, osp, _ = data[other]
                if other == "hatebr" and not spec.get("multilingual"):
                    continue
                if other != "hatebr" and args.train_on == "hatebr":
                    continue
                res["cross"][other] = eval_cross(other, odf.iloc[osp["test"]], predict, k)
            os.makedirs(rdir, exist_ok=True)
            pd.DataFrame({"id": te["id"].values, "y": te["y"].values, "pred": ptest.argmax(1),
                          **{f"p{c}": ptest[:, c] for c in range(k)}}).to_csv(os.path.join(rdir, "preds_test.csv"), index=False)
            json.dump(res, open(os.path.join(rdir, "metrics.json"), "w"), indent=1)
            print(f"   test acc {res['acc']:.4f} macro-F1 {res['macro']['F1']:.4f} hate-F1 {res['F1'][0]:.4f}", file=sys.stderr)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile aggregate_results.py
#!/usr/bin/env python
"""Turn results/ into everything the LaTeX manuscript includes (folder generated/).

    python aggregate_results.py --results results --out generated

Every number in the paper comes out of this script.  Cells whose runs do not exist yet are printed as '--',
so the manuscript always compiles and it is obvious what is still missing.
"""
import argparse
import csv
import glob
import json
import os
import sys
from collections import Counter

import numpy as np
import pandas as pd

import hs_common as C
import run_experiments as R

NAMES = {"tfidf_lr": "TF-IDF + LR", "lstm": "BiLSTM", "distilbert": "DistilBERT", "roberta": "RoBERTa-base",
         "hatebert": "HateBERT", "toxicbert": "toxic-bert", "bert": "BERT-base",
         "bert_unweighted": "BERT, unweighted loss", "bert_focal": "BERT, focal loss",
         "bert_keepsocial": "BERT, hashtags/emoji kept", "xlmr": "XLM-R-base"}
MAIN = ["tfidf_lr", "lstm", "distilbert", "roberta", "hatebert", "toxicbert", "bert"]
ABL = ["bert_unweighted", "bert", "bert_focal", "bert_keepsocial"]
CLS = ["Hate Speech", "Offensive Lang.", "Clean Text"]
DASH = "--"


def key(m):
    return m.replace("_", "")


def load_runs(results, train_on="davidson"):
    runs = {}
    for f in sorted(glob.glob(os.path.join(results, train_on, "*", "seed*", "metrics.json"))):
        r = json.load(open(f))
        runs.setdefault(r["model"], []).append(r)
    # Safety check: every run of one dataset must have been evaluated on the same test set.  Runs from an older
    # version of the code (different row set, e.g. the first keep-social ablation) are excluded and reported.
    sizes = Counter(r["n_test"] for rs in runs.values() for r in rs)
    if len(sizes) > 1:
        ref = sizes.most_common(1)[0][0]
        for m in list(runs):
            bad = [r for r in runs[m] if r["n_test"] != ref]
            if bad:
                print(f"WARNING: {len(bad)} run(s) of '{m}' ({train_on}) were evaluated on a different test set "
                      f"({bad[0]['n_test']} vs {ref} posts) and are excluded; re-run them.", file=sys.stderr)
                runs[m] = [r for r in runs[m] if r["n_test"] == ref]
                if not runs[m]:
                    del runs[m]
    return runs


def stat(vals):
    v = np.asarray(vals, float)
    return v.mean(), (v.std(ddof=1) if len(v) > 1 else 0.0), len(v)


def fmt(vals, d=3, pct=False):
    if not len(vals):
        return DASH
    m, s, n = stat(vals)
    if pct:
        m, s = 100 * m, 100 * s
    return f"{m:.{d}f}" if n == 1 else f"{m:.{d}f}\\,$\\pm$\\,{s:.{d}f}"


def get(runs, m, fn):
    return [fn(r) for r in runs.get(m, [])]


def tabular(cols, rows, align, extra_mid=()):
    out = ["\\begin{tabular}{" + align + "}", "\\toprule", " & ".join(cols) + " \\\\", "\\midrule"]
    for i, row in enumerate(rows):
        if i in extra_mid:
            out.append("\\midrule")
        out.append(" & ".join(row) + " \\\\")
    out += ["\\bottomrule", "\\end{tabular}"]
    return "\n".join(out) + "\n"


def write(out, name, text):
    open(os.path.join(out, name), "w", encoding="utf8").write(text)


# ------------------------------------------------------------------ tables
def table_main(runs, models):
    """Macro-F1 and the Hate Speech class come first; accuracy is last because it is dominated by Offensive Language."""
    rows = []
    for m in models:
        g = lambda f: get(runs, m, f)
        rows.append([NAMES[m], fmt(g(lambda r: r["macro"]["F1"])), fmt(g(lambda r: r["F1"][0])),
                     fmt(g(lambda r: r["P"][0])), fmt(g(lambda r: r["R"][0])),
                     fmt(g(lambda r: r["macro"]["P"])), fmt(g(lambda r: r["macro"]["R"])),
                     fmt(g(lambda r: r["weighted"]["F1"])), fmt(g(lambda r: r["acc"]))])
    return tabular(["Model", "Macro-F1", "Hate F1", "Hate P", "Hate R", "Macro-P", "Macro-R", "Weighted-F1", "Accuracy"],
                   rows, "lcccccccc")


def table_perclass(runs, models):
    rows, mids = [], []
    for m in models:
        if rows:
            mids.append(len(rows))
        for c in range(3):
            g = lambda f: get(runs, m, f)
            rows.append([NAMES[m] if c == 0 else "", CLS[c], fmt(g(lambda r: r["P"][c])), fmt(g(lambda r: r["R"][c])),
                         fmt(g(lambda r: r["F1"][c])), fmt(g(lambda r: r["spec"][c]))])
    return tabular(["Model", "Class", "Precision", "Recall", "F1", "Specificity"], rows, "llcccc", mids)


def table_ablation(runs):
    rows = []
    for m in ABL:
        g = lambda f: get(runs, m, f)
        rows.append([NAMES[m], fmt(g(lambda r: r["acc"])), fmt(g(lambda r: r["macro"]["F1"])),
                     fmt(g(lambda r: r["P"][0])), fmt(g(lambda r: r["R"][0])), fmt(g(lambda r: r["F1"][0]))])
    return tabular(["Variant", "Accuracy", "Macro-F1", "Hate P", "Hate R", "Hate F1"], rows, "lccccc")


def table_cost(runs, models):
    rows = []
    for m in models:
        rs = runs.get(m, [])
        if not rs:
            rows.append([NAMES[m]] + [DASH] * 5)
            continue
        gpu = Counter(r.get("env", {}).get("gpu", "?") for r in rs).most_common(1)[0][0]
        rows.append([NAMES[m], f"{np.mean([r['cost']['n_params'] for r in rs]) / 1e6:.2f}",
                     f"{np.mean([r['cost']['epochs_run'] for r in rs]):.1f}",
                     f"{np.mean([r['cost']['train_seconds'] for r in rs]) / 60:.1f}",
                     f"{np.mean([r['cost']['infer_ms_per_post'] for r in rs]):.2f}", gpu.replace("_", "\\_")])
    return tabular(["Model", "Parameters (M)", "Epochs", "Training (min)", "Inference (ms/post)", "Device"], rows, "lccccl")


def table_config():
    hp = R.HP
    rows = [["Split", "stratified 70\\,/\\,10\\,/\\,20\\,\\% (train\\,/\\,validation\\,/\\,test), split seed " + str(C.SPLIT_SEED)
             + ", identical for all models"],
            ["Training seeds", ", ".join(map(str, R.DEFAULT_SEEDS)) + " (five runs per model)"],
            ["Maximum sequence length", str(hp["max_len"]) + " tokens"],
            ["Optimizer", "AdamW, learning rate $2\\times10^{-5}$, weight decay " + str(hp["weight_decay"])],
            ["Batch size", str(hp["batch_size"])],
            ["Schedule", "linear decay, warm-up over " + str(int(100 * hp["warmup_ratio"])) + "\\,\\% of the steps"],
            ["Epochs / early stopping", f"at most {hp['max_epochs']} epochs; best epoch by validation macro-F1; stop after {hp['patience']} epochs without improvement"],
            ["Regularization", "default dropout of each pre-trained model (0.1); gradient clipping at " + str(hp["grad_clip"])],
            ["Class weights", "$w_c = n / (K\\,n_c)$ from the training set ($K=3$); used in the weighted cross-entropy loss"],
            ["Focal loss variant", "$\\gamma = " + str(hp["focal_gamma"]) + "$, $\\alpha_c = w_c$"],
            ["BiLSTM baseline", f"embedding {hp['lstm']['emb']}, hidden {hp['lstm']['hidden']} (bidirectional), dropout {hp['lstm']['dropout']}, "
                                f"{hp['lstm']['epochs']} epochs, Adam ({hp['lstm']['lr']}), trained from scratch"],
            ["TF-IDF baseline", "word 1--2-grams, minimum document frequency 2, sublinear TF, logistic regression ($C=1$, balanced class weights)"],
            ["Precision", "mixed precision (fp16) on GPU"]]
    out = ["\\begin{tabular}{@{}lp{0.68\\linewidth}@{}}", "\\toprule", "Setting & Value \\\\", "\\midrule"]
    out += [" & ".join(r) + " \\\\" for r in rows]
    out += ["\\bottomrule", "\\end{tabular}"]
    return "\n".join(out) + "\n"


def table_cross(cache_results, results):
    dav, hx = load_runs(results, "davidson"), load_runs(results, "hatexplain")
    rows = []
    for m in ["tfidf_lr", "lstm", "distilbert", "roberta", "hatebert", "toxicbert", "bert"]:
        rd = dav.get(m, [])
        rows.append([NAMES[m] + " (train: Davidson)", fmt([r["macro"]["F1"] for r in rd]),
                     fmt([r["cross"]["hatexplain"]["macro"]["F1"] for r in rd if "hatexplain" in r["cross"]]),
                     fmt([r["cross"]["hatexplain"]["F1"][0] for r in rd if "hatexplain" in r["cross"]])])
    for m in ["bert", "hatebert"]:
        rh = hx.get(m, [])
        rows.append([NAMES[m] + " (train: HateXplain)", fmt([r["cross"]["davidson"]["macro"]["F1"] for r in rh if "davidson" in r["cross"]]),
                     fmt([r["macro"]["F1"] for r in rh]), fmt([r["F1"][0] for r in rh])])
    return tabular(["Model and training data", "Davidson test macro-F1", "HateXplain test macro-F1", "HateXplain test Hate F1"],
                   rows, "lccc", (7,))


def table_multilingual(results):
    dav, br = load_runs(results, "davidson"), load_runs(results, "hatebr")
    rows = []
    rd = dav.get("xlmr", [])
    z = [r["cross"]["hatebr"] for r in rd if "hatebr" in r["cross"]]
    rows.append(["XLM-R, trained on Davidson (English), zero-shot", fmt([c["acc"] for c in z]), fmt([c["F1"][1] for c in z]),
                 fmt([c["macro"]["F1"] for c in z])])
    rb = br.get("xlmr", [])
    rows.append(["XLM-R, trained on HateBR (Portuguese)", fmt([r["acc"] for r in rb]), fmt([r["F1"][1] for r in rb]),
                 fmt([r["macro"]["F1"] for r in rb])])
    return tabular(["Model", "Accuracy", "Offensive F1", "Macro-F1"], rows, "lccc")


# ------------------------------------------------------------------ statistics
def macro_f1_from(y, p, idx, k=3):
    cm = np.bincount(y[idx] * k + p[idx], minlength=k * k).reshape(k, k).astype(float)
    tp = np.diag(cm)
    P = np.where(cm.sum(0) > 0, tp / np.maximum(cm.sum(0), 1), 0)
    R = np.where(cm.sum(1) > 0, tp / np.maximum(cm.sum(1), 1), 0)
    F = np.where(P + R > 0, 2 * P * R / np.maximum(P + R, 1e-12), 0)
    return F.mean()


def load_preds(results, m, train_on="davidson"):
    fs = sorted(glob.glob(os.path.join(results, train_on, m, "seed*", "preds_test.csv")))
    if not fs:
        return None, None, None
    dfs = [pd.read_csv(f) for f in fs]
    return dfs[0]["id"].values, dfs[0]["y"].values, np.stack([d["pred"].values for d in dfs])


def table_stats(runs, results, ref="bert", B=2000, seed=0):
    from scipy import stats
    rows = []
    rid, ry, rp = load_preds(results, ref)
    for m in ["tfidf_lr", "lstm", "distilbert", "roberta", "hatebert", "toxicbert"]:
        if rp is None:
            rows.append([f"{NAMES[ref]} vs {NAMES[m]}", DASH, DASH, DASH])
            continue
        mid, my, mp = load_preds(results, m)
        if mp is None or not np.array_equal(mid, rid):
            rows.append([f"{NAMES[ref]} vs {NAMES[m]}", DASH, DASH, DASH])
            continue
        rng = np.random.RandomState(seed)
        n = len(ry)
        diffs = np.empty(B)
        for b in range(B):
            idx = rng.randint(0, n, n)
            diffs[b] = (np.mean([macro_f1_from(ry, p, idx) for p in rp]) - np.mean([macro_f1_from(ry, p, idx) for p in mp]))
        d0 = np.mean([macro_f1_from(ry, p, np.arange(n)) for p in rp]) - np.mean([macro_f1_from(ry, p, np.arange(n)) for p in mp])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        a = [r["macro"]["F1"] for r in runs.get(ref, [])]
        b_ = [r["macro"]["F1"] for r in runs.get(m, [])]
        pval = DASH
        if len(a) > 1 and len(b_) > 1:
            pval = f"{stats.ttest_ind(a, b_, equal_var=False).pvalue:.3f}"
            if pval == "0.000":
                pval = "$<$0.001"
        rows.append([f"{NAMES[ref]} vs {NAMES[m]}", f"{d0:+.3f}", f"[{lo:+.3f}, {hi:+.3f}]", pval])
    return tabular(["Comparison (macro-F1)", "Difference", "95\\% bootstrap CI", "Welch $p$ (seeds)"], rows, "lccc")


def table_stats_ablation(runs, results, ref="bert", B=2000, seed=0):
    """Variant minus BERT-base (positive: the variant is higher), same test posts, paired bootstrap over posts."""
    from scipy import stats
    rows, macros = [], {}
    rid, ry, rp = load_preds(results, ref)
    for m in ["bert_unweighted", "bert_focal", "bert_keepsocial"]:
        mid, my, mp = load_preds(results, m) if rp is not None else (None, None, None)
        if mp is None or not np.array_equal(mid, rid) or not np.array_equal(my, ry):
            rows.append([f"{NAMES[m]} vs {NAMES[ref]}", DASH, DASH, DASH])
            for suf in ("Diff", "Lo", "Hi", "P"):
                macros[f"abl{key(m)}{suf}"] = DASH
            continue
        rng = np.random.RandomState(seed)
        n = len(ry)
        diffs = np.empty(B)
        for b in range(B):
            idx = rng.randint(0, n, n)
            diffs[b] = np.mean([macro_f1_from(ry, p, idx) for p in mp]) - np.mean([macro_f1_from(ry, p, idx) for p in rp])
        d0 = np.mean([macro_f1_from(ry, p, np.arange(n)) for p in mp]) - np.mean([macro_f1_from(ry, p, np.arange(n)) for p in rp])
        lo, hi = np.percentile(diffs, [2.5, 97.5])
        a = [r["macro"]["F1"] for r in runs.get(m, [])]
        b_ = [r["macro"]["F1"] for r in runs.get(ref, [])]
        pval = DASH
        if len(a) > 1 and len(b_) > 1:
            pval = f"{stats.ttest_ind(a, b_, equal_var=False).pvalue:.3f}"
            if pval == "0.000":
                pval = "$<$0.001"
        rows.append([f"{NAMES[m]} vs {NAMES[ref]}", f"{d0:+.3f}", f"[{lo:+.3f}, {hi:+.3f}]", pval])
        macros[f"abl{key(m)}Diff"] = f"{d0:+.3f}"
        macros[f"abl{key(m)}Lo"] = f"{lo:+.3f}"
        macros[f"abl{key(m)}Hi"] = f"{hi:+.3f}"
        macros[f"abl{key(m)}P"] = pval
    return tabular(["Comparison (macro-F1)", "Difference", "95\\% bootstrap CI", "Welch $p$ (seeds)"], rows, "lccc"), macros


def cross_macros(runs, results):
    """Numbers quoted in the text about external tests."""
    m = {}
    hx, br = load_runs(results, "hatexplain"), load_runs(results, "hatebr")
    def mean(v): return float(np.mean(v))
    rd = runs.get("bert", [])
    z = [r["cross"]["hatexplain"]["macro"]["F1"] for r in rd if "hatexplain" in r["cross"]]
    if z: m["xBertHxMacro"] = f"{mean(z):.3f}"
    lo = [mean([r["cross"]["hatexplain"]["macro"]["F1"] for r in runs[k] if "hatexplain" in r["cross"]]) for k in MAIN if runs.get(k)]
    if lo: m["xHxMacroMin"], m["xHxMacroMax"] = f"{min(lo):.2f}", f"{max(lo):.2f}"
    h = [r["cross"]["davidson"]["macro"]["F1"] for r in hx.get("bert", []) if "davidson" in r["cross"]]
    if h: m["xHxToDavMacro"] = f"{mean(h):.3f}"
    h2 = [r["macro"]["F1"] for r in hx.get("bert", [])]
    if h2: m["xHxInMacro"] = f"{mean(h2):.3f}"
    zz = [r["cross"]["hatebr"]["macro"]["F1"] for r in runs.get("xlmr", []) if "hatebr" in r["cross"]]
    if zz: m["xBrZeroMacro"] = f"{mean(zz):.3f}"
    zi = [r["macro"]["F1"] for r in br.get("xlmr", [])]
    if zi: m["xBrInMacro"] = f"{mean(zi):.3f}"
    return m


# ------------------------------------------------------------------ error analysis
def error_analysis(results, cache, out, model="bert"):
    ids, y, P = load_preds(results, model)
    empty = tabular(["Measure", "Hate detected", "Hate missed", "False alarms"],
                    [[m_, DASH, DASH, DASH] for m_ in ["Posts", "Annotator agreement", "Unanimous annotators (\\%)",
                                                        "Vote share for Hate (\\%)", "Length (tokens)",
                                                        "Offensive-leaning vocabulary (\\%)", "Missed by all seeds (\\%)"]], "lccc")
    if P is None or not os.path.exists(os.path.join(cache, "davidson_labeled_data.csv")):
        return empty, {}
    from scipy import stats
    df, _ = C.load_davidson(cache)
    df = df.set_index("id").loc[ids].reset_index()
    tr_idx, _, _ = C.split_indices(C.load_davidson(cache)[0]["y"].values)
    full = C.load_davidson(cache)[0]
    train = full.iloc[tr_idx]
    # offensive-leaning vocabulary: highest smoothed log-odds Offensive vs Hate in the training set
    ch = Counter(w for t in train[train.y == 0]["text"] for w in t.split())
    co = Counter(w for t in train[train.y == 1]["text"] for w in t.split())
    nh, no = sum(ch.values()), sum(co.values())
    vocab = {w for w in set(ch) | set(co) if ch[w] + co[w] >= 20}
    lo = {w: np.log((co[w] + 0.5) / (no + 0.5 * len(vocab))) - np.log((ch[w] + 0.5) / (nh + 0.5 * len(vocab))) for w in vocab}
    offvocab = set(sorted(lo, key=lo.get, reverse=True)[:150])
    frac_not_hate = (P != 0).mean(0)
    hate, off = y == 0, y == 1
    missed = hate & (frac_not_hate >= 0.5)
    found = hate & (frac_not_hate < 0.5)
    alarm = off & ((P == 0).mean(0) >= 0.5)
    df["agree"] = df[["v_hate", "v_off", "v_clean"]].max(axis=1) / df["n_votes"]
    df["unan"] = (df[["v_hate", "v_off", "v_clean"]].max(axis=1) == df["n_votes"]).astype(float)
    df["hshare"] = df["v_hate"] / df["n_votes"]
    df["len"] = df["text"].str.split().str.len()
    df["offshare"] = [np.mean([w in offvocab for w in t.split()]) if t.split() else 0 for t in df["text"]]
    groups = {"found": found, "missed": missed, "alarm": alarm}

    def col(name, f, d=2, pct=False):
        cells = []
        for g in ("found", "missed", "alarm"):
            v = f(df[groups[g]])
            cells.append(DASH if np.isnan(v) else (f"{100 * v:.1f}" if pct else f"{v:.{d}f}"))
        return [name] + cells
    rows = [["Posts"] + [str(int(groups[g].sum())) for g in ("found", "missed", "alarm")],
            col("Annotator agreement", lambda d: d["agree"].mean()),
            col("Unanimous annotators (\\%)", lambda d: d["unan"].mean(), pct=True),
            col("Vote share for Hate (\\%)", lambda d: d["hshare"].mean(), pct=True),
            col("Length (tokens)", lambda d: d["len"].mean(), 1),
            col("Offensive-leaning vocabulary (\\%)", lambda d: d["offshare"].mean(), pct=True)]
    all_missed = float(np.mean((P[:, missed] != 0).all(0)) * 100) if missed.sum() else 0.0
    rows.append(["Missed by all seeds (\\%)", DASH, f"{all_missed:.1f}", DASH])
    macros = {}
    if found.sum() > 1 and missed.sum() > 1:
        macros["errAgreeP"] = f"{stats.mannwhitneyu(df[found]['agree'], df[missed]['agree']).pvalue:.3g}"
        macros["errOffP"] = f"{stats.mannwhitneyu(df[found]['offshare'], df[missed]['offshare']).pvalue:.3g}"
    macros["errAgreeFound"] = f"{df[found]['agree'].mean():.2f}"
    macros["errAgreeMissed"] = f"{df[missed]['agree'].mean():.2f}"
    macros["errUnanFound"] = f"{100 * df[found]['unan'].mean():.0f}"
    macros["errUnanMissed"] = f"{100 * df[missed]['unan'].mean():.0f}"
    macros["errOffFound"] = f"{100 * df[found]['offshare'].mean():.1f}"
    macros["errOffMissed"] = f"{100 * df[missed]['offshare'].mean():.1f}"
    macros["errAllSeeds"] = f"{all_missed:.0f}"
    macros["errAlarm"] = str(int(alarm.sum()))
    macros["errMissed"] = str(int(missed.sum()))
    macros["errHate"] = str(int(hate.sum()))
    # manual-coding sheet: 40 missed Hate posts + 20 false alarms, fixed random sample
    rs = np.random.RandomState(7)
    pick = list(rs.choice(np.where(missed)[0], size=min(40, int(missed.sum())), replace=False)) + \
        list(rs.choice(np.where(alarm)[0], size=min(20, int(alarm.sum())), replace=False))
    sheet = df.iloc[pick][["id", "raw", "text", "v_hate", "v_off", "v_clean"]].copy()
    sheet.insert(1, "type", ["missed_hate"] * min(40, int(missed.sum())) + ["false_alarm"] * min(20, int(alarm.sum())))
    sheet["code"] = ""
    sheet["notes"] = ""
    sheet.to_csv(os.path.join(out, "errors_to_code.csv"), index=False)
    return tabular(["Measure", "Hate detected", "Hate missed", "False alarms"], rows, "lccc"), macros


CODES = ["implicit_hate", "sarcasm_irony", "quoted_or_reported", "reclaimed_ingroup", "label_ambiguous",
         "explicit_but_untargeted", "other"]
CODE_NAMES = {"implicit_hate": "Implicit hate (no explicit slur)", "sarcasm_irony": "Sarcasm or irony",
              "quoted_or_reported": "Quoted or reported speech", "reclaimed_ingroup": "Reclaimed / in-group use",
              "label_ambiguous": "Label debatable", "explicit_but_untargeted": "Explicit insult, no protected target",
              "other": "Other"}


def table_error_codes(out):
    p = os.path.join(out, "errors_coded.csv")
    rows = []
    if os.path.exists(p):
        d = pd.read_csv(p)
        d = d[d["code"].isin(CODES)]
        for t in ("missed_hate", "false_alarm"):
            sub = d[d["type"] == t]
            for c in CODES:
                rows.append([("Missed Hate" if t == "missed_hate" else "False alarms") if c == CODES[0] else "",
                             CODE_NAMES[c], str(int((sub["code"] == c).sum())),
                             f"{100 * (sub['code'] == c).mean():.0f}" if len(sub) else DASH])
    else:
        rows = [["Missed Hate" if i == 0 else "", CODE_NAMES[c], DASH, DASH] for i, c in enumerate(CODES)]
    return tabular(["Error type", "Category", "Posts", "\\%"], rows, "llcc", (len(CODES),))


# ------------------------------------------------------------------ figures
def figures(runs, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    def draw(ax, m):
        rs = runs.get(m, [])
        if not rs:
            ax.text(0.5, 0.5, "not run yet", ha="center", va="center"); ax.axis("off"); return
        cm = np.mean([np.array(r["cm"], float) for r in rs], axis=0)
        pr = cm / cm.sum(1, keepdims=True)
        ax.imshow(pr, cmap="Blues", vmin=0, vmax=1)
        for i in range(3):
            for j in range(3):
                ax.text(j, i, f"{100 * pr[i, j]:.1f}%\n(n={cm[i, j]:.0f})", ha="center", va="center", fontsize=7,
                        color="white" if pr[i, j] > 0.5 else "black")
        ax.set_xticks(range(3)); ax.set_yticks(range(3))
        ax.set_xticklabels(["Hate", "Offensive", "Clean"], fontsize=8); ax.set_yticklabels(["Hate", "Offensive", "Clean"], fontsize=8)
        ax.set_xlabel("Predicted", fontsize=8); ax.set_ylabel("True label", fontsize=8)
        ax.set_title(NAMES[m], fontsize=9)
    for m in MAIN:
        fig, ax = plt.subplots(figsize=(3.3, 3.0)); draw(ax, m); fig.tight_layout()
        fig.savefig(os.path.join(out, f"figure_cm_{key(m)}.pdf")); plt.close(fig)
    fig, axes = plt.subplots(2, 4, figsize=(11, 5.2))
    for ax, m in zip(axes.ravel(), MAIN):
        draw(ax, m)
    for ax in axes.ravel()[len(MAIN):]:
        ax.axis("off")
    fig.tight_layout(); fig.savefig(os.path.join(out, "figure_cm_all.pdf")); plt.close(fig)


def figure_class_dist(cache, out):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    p = os.path.join(cache, "davidson_labeled_data.csv")
    if not os.path.exists(p):
        return
    df, _ = C.load_davidson(cache)
    cnt = df["y"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(4.6, 3.0))
    bars = ax.bar(["Hate\nSpeech", "Offensive\nLanguage", "Clean\nText"], cnt.values, color=["#d62728", "#f2c744", "#2ca02c"])
    for b, v in zip(bars, cnt.values):
        ax.text(b.get_x() + b.get_width() / 2, v + 250, f"{v:,}\n({100 * v / cnt.sum():.1f}%)", ha="center", fontsize=8)
    ax.set_ylabel("Posts", fontsize=9); ax.set_ylim(0, cnt.max() * 1.18)
    ax.tick_params(axis="x", labelsize=10); ax.tick_params(axis="y", labelsize=9)  # horizontal, readable labels
    fig.tight_layout(); fig.savefig(os.path.join(out, "figure_class_dist.pdf")); plt.close(fig)


# ------------------------------------------------------------------ macros
def numbers(runs, cache, results, out, extra):
    lines = ["\\providecommand{\\res}[2]{\\csname res@#1@#2\\endcsname}",
             "\\providecommand{\\setres}[3]{\\expandafter\\gdef\\csname res@#1@#2\\endcsname{#3}}"]

    def setres(k, name, v):
        lines.append(f"\\setres{{{k}}}{{{name}}}{{{v}}}")
    metrics = {"Acc": (lambda r: r["acc"], True, 1), "MacroP": (lambda r: r["macro"]["P"], False, 3),
               "MacroR": (lambda r: r["macro"]["R"], False, 3), "MacroF": (lambda r: r["macro"]["F1"], False, 3),
               "WeightedF": (lambda r: r["weighted"]["F1"], False, 3), "HateP": (lambda r: r["P"][0], False, 3),
               "HateR": (lambda r: r["R"][0], False, 3), "HateF": (lambda r: r["F1"][0], False, 3),
               "OffF": (lambda r: r["F1"][1], False, 3), "CleanF": (lambda r: r["F1"][2], False, 3)}
    for m in R.MODELS:
        k = key(m)
        rs = runs.get(m, [])
        setres(k, "name", NAMES[m])
        setres(k, "n", len(rs) if rs else DASH)
        for name, (f, pct, d) in metrics.items():
            if rs:
                mean, sd, _ = stat([f(r) for r in rs])
                setres(k, name, f"{100 * mean:.{d}f}" if pct else f"{mean:.{d}f}")
                setres(k, name + "Sd", f"{100 * sd:.{d}f}" if pct else f"{sd:.{d}f}")
            else:
                setres(k, name, DASH); setres(k, name + "Sd", DASH)
        if rs:
            setres(k, "Params", f"{np.mean([r['cost']['n_params'] for r in rs]) / 1e6:.0f}")
            setres(k, "TrainMin", f"{np.mean([r['cost']['train_seconds'] for r in rs]) / 60:.1f}")
            setres(k, "InferMs", f"{np.mean([r['cost']['infer_ms_per_post'] for r in rs]):.2f}")
        else:
            for name in ("Params", "TrainMin", "InferMs"):
                setres(k, name, DASH)
    have = [m for m in MAIN if runs.get(m)]
    if have:
        b = max(have, key=lambda m: np.mean([r["macro"]["F1"] for r in runs[m]]))
        h = max(have, key=lambda m: np.mean([r["F1"][0] for r in runs[m]]))
        lines.append(f"\\gdef\\bestModel{{{NAMES[b]}}}\\gdef\\bestMacroF{{{np.mean([r['macro']['F1'] for r in runs[b]]):.3f}}}")
        lines.append(f"\\gdef\\bestHateModel{{{NAMES[h]}}}\\gdef\\bestHateF{{{np.mean([r['F1'][0] for r in runs[h]]):.3f}}}")
        seeds = sorted({r["seed"] for m in have for r in runs[m]})
        lines.append(f"\\gdef\\nSeeds{{{max(len(runs[m]) for m in have)}}}")
    else:
        lines += ["\\gdef\\bestModel{--}\\gdef\\bestMacroF{--}\\gdef\\bestHateModel{--}\\gdef\\bestHateF{--}\\gdef\\nSeeds{5}"]
    # data macros
    info = next((runs[m][0]["data_info"] for m in R.MODELS if runs.get(m)), None)
    p = os.path.join(cache, "davidson_labeled_data.csv")
    if info and os.path.exists(p):
        df, _ = C.load_davidson(cache)
        cnt = df["y"].value_counts().sort_index()
        n = len(df)
        vals = {"dataRowsFile": info["rows_in_file"], "dataEmpty": info["rows_empty_after_cleaning"], "dataN": n,
                "dataHate": int(cnt[0]), "dataOff": int(cnt[1]), "dataClean": int(cnt[2]),
                "dataHatePct": f"{100 * cnt[0] / n:.2f}", "dataOffPct": f"{100 * cnt[1] / n:.2f}",
                "dataCleanPct": f"{100 * cnt[2] / n:.2f}", "dataSha": info["file_sha256"]}
        r0 = next(runs[m][0] for m in R.MODELS if runs.get(m))
        vals.update(dataTrain=r0["n_train"], dataVal=r0["n_val"], dataTest=r0["n_test"])
    else:
        vals = {k: DASH for k in ["dataRowsFile", "dataEmpty", "dataN", "dataHate", "dataOff", "dataClean", "dataHatePct",
                                  "dataOffPct", "dataCleanPct", "dataSha", "dataTrain", "dataVal", "dataTest"]}
        vals["dataSha"] = C.DAVIDSON_SHA256
    if os.path.exists(p):
        df_, _ = C.load_davidson(cache)
        _, _, te_ = C.split_indices(df_["y"].values)
        yt = df_["y"].values[te_]
        cnt_t = np.bincount(yt, minlength=3)
        cm_maj = np.zeros((3, 3), dtype=int)
        cm_maj[:, 1] = cnt_t  # a classifier that always says "Offensive Language"
        mm = C.cm_metrics(cm_maj)
        vals.update(testHate=int(cnt_t[0]), testOff=int(cnt_t[1]), testClean=int(cnt_t[2]),
                    majAcc=f"{100 * mm['acc']:.1f}", majMacroF=f"{mm['macro']['F1']:.2f}")
    for k, v in vals.items():
        lines.append(f"\\gdef\\{k}{{{v}}}")
    for k, v in extra.items():
        lines.append(f"\\gdef\\{k}{{{v}}}")
    # hardware / software (taken from the transformer runs)
    env = next((runs["bert"][0].get("env", {}) for _ in [0] if runs.get("bert")), None) or {}
    for k, v in {"envGpu": env.get("gpu", DASH), "envTorch": env.get("torch", DASH),
                 "envTransformers": env.get("transformers", DASH), "envPython": env.get("python", DASH)}.items():
        lines.append(f"\\gdef\\{k}{{{str(v).replace('_', ' ')}}}")
    write(out, "numbers.tex", "\n".join(lines) + "\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    ap.add_argument("--out", default="generated")
    ap.add_argument("--cache", default="data_cache")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    runs = load_runs(a.results)
    write(a.out, "table_main.tex", table_main(runs, MAIN))
    write(a.out, "table_perclass_bert.tex", table_perclass(runs, ["bert"]))
    write(a.out, "table_perclass_all.tex", table_perclass(runs, MAIN))
    write(a.out, "table_ablation.tex", table_ablation(runs))
    write(a.out, "table_cost.tex", table_cost(runs, MAIN))
    write(a.out, "table_config.tex", table_config())
    write(a.out, "table_cross.tex", table_cross(a.cache, a.results))
    write(a.out, "table_multilingual.tex", table_multilingual(a.results))
    write(a.out, "table_stats.tex", table_stats(runs, a.results))
    tab_abl, abl_macros = table_stats_ablation(runs, a.results)
    write(a.out, "table_stats_ablation.tex", tab_abl)
    tab, extra = error_analysis(a.results, a.cache, a.out)
    extra.update(abl_macros)
    extra.update(cross_macros(runs, a.results))
    write(a.out, "table_errors.tex", tab)
    write(a.out, "table_error_codes.tex", table_error_codes(a.out))
    figures(runs, a.out)
    figure_class_dist(a.cache, a.out)
    numbers(runs, a.cache, a.results, a.out, extra)
    print("wrote", sorted(os.listdir(a.out)))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile check_claims.py
#!/usr/bin/env python
"""Check the qualitative statements the manuscript makes against the actual results.

    python check_claims.py --results results

The prose in the Discussion and Conclusion was written so that it does not depend on which model wins.  The
statements below ARE asserted in the text, though; if one prints FAIL, edit the matching sentence.
"""
import argparse
import numpy as np
import aggregate_results as A


def mean(runs, m, f):
    return np.mean([f(r) for r in runs[m]]) if runs.get(m) else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    a = ap.parse_args()
    runs = A.load_runs(a.results)
    hx = A.load_runs(a.results, "hatexplain")
    checks = []

    def add(label, ok, sentence):
        checks.append((label, ok, sentence))

    have = [m for m in A.MAIN if runs.get(m)]
    add("C1 Hate Speech has the lowest F1 of the three classes for every model",
        all(min(range(3), key=lambda c: mean(runs, m, lambda r: r["F1"][c])) == 0 for m in have) if have else None,
        "Discussion, paragraph 1")
    tf = [m for m in ("distilbert", "roberta", "hatebert", "toxicbert", "bert") if runs.get(m)]
    add("C2 every transformer beats TF-IDF+LR and BiLSTM on macro-F1",
        all(mean(runs, m, lambda r: r["macro"]["F1"]) > max(mean(runs, b, lambda r: r["macro"]["F1"]) for b in ("tfidf_lr", "lstm") if runs.get(b))
            for m in tf) if tf and runs.get("tfidf_lr") and runs.get("lstm") else None, "Results / Discussion")
    if runs.get("bert") and runs.get("bert_unweighted"):
        add("C3 class weighting raises Hate Speech recall relative to unweighted loss",
            mean(runs, "bert", lambda r: r["R"][0]) > mean(runs, "bert_unweighted", lambda r: r["R"][0]), "Ablation paragraph")
        add("C4 class weighting does not lower macro-F1 relative to unweighted loss",
            mean(runs, "bert", lambda r: r["macro"]["F1"]) >= mean(runs, "bert_unweighted", lambda r: r["macro"]["F1"]) - 0.005,
            "Ablation paragraph")
    else:
        add("C3/C4 class-weighting ablation", None, "Ablation paragraph")
    if runs.get("bert"):
        add("C5 most missed Hate posts are predicted as Offensive Language, not Clean",
            mean(runs, "bert", lambda r: r["cm"][0][1]) > mean(runs, "bert", lambda r: r["cm"][0][2]), "Error analysis")
    xs = [m for m in ("bert", "roberta", "hatebert", "distilbert", "toxicbert", "lstm", "tfidf_lr")
          if runs.get(m) and all("hatexplain" in r["cross"] for r in runs[m])]
    add("C6 macro-F1 on HateXplain (zero-shot) is below in-domain macro-F1 for every model",
        all(mean(runs, m, lambda r: r["cross"]["hatexplain"]["macro"]["F1"]) < mean(runs, m, lambda r: r["macro"]["F1"]) for m in xs) if xs else None,
        "Generalization paragraph")
    if have:
        acc = [mean(runs, m, lambda r: r["acc"]) for m in have]
        hf = [mean(runs, m, lambda r: r["F1"][0]) for m in have]
        add("C7 the spread of accuracy across models is smaller than the spread of Hate Speech F1",
            (max(acc) - min(acc)) < (max(hf) - min(hf)), "Discussion, paragraph 1")
    for label, ok, where in checks:
        print(f"{'PASS' if ok else ('n/a ' if ok is None else 'FAIL')}  {label}   [{where}]")


if __name__ == "__main__":
    main()

In [ ]:
import hs_common as C, numpy as np
df, info = C.load_davidson(WORK + "/data_cache")
tr, va, te = C.split_indices(df["y"].values)
print(info); print("train/val/test =", len(tr), len(va), len(te)); print("class counts:", np.bincount(df["y"].values))
assert info["file_sha256"] == C.DAVIDSON_SHA256, "Davidson file differs from the one used in the paper"
assert (len(tr), len(va), len(te)) == (17334, 2477, 4953), "split sizes differ from the paper"
print(df["raw"].iloc[5], "->", df["text"].iloc[5])

## 2. Main comparison (RQ1, RQ2), five seeds
TF-IDF+LR, BiLSTM, DistilBERT, RoBERTa-base, HateBERT, toxic-bert, BERT-base. Each run also tests the Davidson-trained model zero-shot on HateXplain and records parameters, training/inference time and GPU memory.

In [ ]:
MAIN = "tfidf_lr lstm bert distilbert roberta hatebert toxicbert"
!python run_experiments.py --out "$WORK/results" --cache "$WORK/data_cache" --models $MAIN --seeds {" ".join(map(str, SEEDS))} 2>&1 | grep --line-buffered -v "Warning\|warn" 

## 3. Ablations on BERT-base (RQ3)
Unweighted loss, focal loss (gamma 2), and keeping hashtags/emoji.

In [ ]:
ABL = "bert_unweighted bert_focal bert_keepsocial"
!python run_experiments.py --out "$WORK/results" --cache "$WORK/data_cache" --models $ABL --seeds {" ".join(map(str, SEEDS))} 2>&1 | grep --line-buffered -v "Warning\|warn" 

## 4. Generalisation (RQ4)
HateXplain (train + zero-shot on Davidson) and HateBR (Portuguese, XLM-R). Three seeds.

In [ ]:
G = " ".join(map(str, SEEDS[:3]))
!python run_experiments.py --out "$WORK/results" --cache "$WORK/data_cache" --train-on hatexplain --models bert hatebert --seeds $G 2>&1 | grep --line-buffered -v "Warning\|warn"
!python run_experiments.py --out "$WORK/results" --cache "$WORK/data_cache" --models xlmr --seeds $G 2>&1 | grep --line-buffered -v "Warning\|warn"
!python run_experiments.py --out "$WORK/results" --cache "$WORK/data_cache" --train-on hatebr --models xlmr --seeds $G 2>&1 | grep --line-buffered -v "Warning\|warn" 

## 5. Tables, figures, numbers and claim checks

In [ ]:
!python aggregate_results.py --results "$WORK/results" --out "$WORK/generated" --cache "$WORK/data_cache"
!ls "$WORK/generated" 

In [ ]:
!python check_claims.py --results "$WORK/results" 

In [ ]:
import pandas as pd
from IPython.display import display
rows = []
for p in sorted(glob.glob(WORK + "/results/davidson/*/seed*/metrics.json")):
    r = json.load(open(p)); rows.append(dict(model=r["model"], seed=r["seed"], acc=r["acc"], macroF1=r["macro"]["F1"], hateF1=r["F1"][0]))
display(pd.DataFrame(rows).groupby("model").agg(["mean", "std"]).round(3))

## 6. Package for the paper
Creates `/kaggle/working/hs_results_and_generated.zip` (metrics + predictions, tables, figures; no raw data). After a *Commit* run, download it from the notebook's **Output** tab; in an interactive session download it from the file panel on the right (`/kaggle/working`) before closing. Then upload it to Claude.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/hs_results_and_generated", "zip", WORK, ".")
print(os.path.getsize("/kaggle/working/hs_results_and_generated.zip") / 1e6, "MB -> /kaggle/working/hs_results_and_generated.zip")